# Day 14 · 习题集 2 与模拟面试（一）

**心理大模型评测与 Agent** · 复习与习题 · 建议用时 4 小时

**今天的目标**

- 独立完成 5 道综合题，覆盖第 8 到 13 天
- 完成一次 30 分钟模拟面试并复盘
- 定稿项目的题目与 rubric

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

# 习题集 2

情境不变：面向香港老年人的心理支持聊天机器人。这次你负责评测流程本身。

## 题 1：短版评测集

在 A 半模型上挑 12 道题（目标区间 θ ∈ [0, 2])，在 B 半上计算短版与全版通过率的相关。

In [2]:
from course.data import simulate_vignette_matrix
from course.irt import fit_2pl, select_items_greedy
u, theta, a_true, b_true, items = simulate_vignette_matrix(n_models=100, n_items=50, seed=140)
perm = np.random.default_rng(1).permutation(100); A, B = perm[:50], perm[50:]
fA = fit_2pl(u[A])
_ch = select_items_greedy(fA["a"], fA["b"], np.linspace(0, 2, 21), 12)   # 题 1 要用的 12 道题
q1 = None
check("题 1 B 半相关", q1, np.corrcoef(u[B][:, _ch].mean(1), u[B].mean(1))[0, 1], tol=1e-6)

[ 待完成 ] 题 1 B 半相关


## 题 2：前两名能否区分

In [3]:
from course.bt import simulate_pairs, fit_bradley_terry, bootstrap_bt_ranks
votes2 = [(w, l) for w, l, _ in simulate_pairs(np.array([0.5, 0.42, 0.1, -0.2, -0.5]), 400, seed=141)]
ranks2, _ = bootstrap_bt_ranks(votes2, 5, n_boot=400, seed=2)
lo = np.quantile(ranks2, .025, 0); hi = np.quantile(ranks2, .975, 0)
q2 = None   # 前两名(系统 0 和 1)排名区间是否重叠?True/False
check("题 2", q2, bool(hi[0] >= lo[1] and hi[1] >= lo[0]))

[ 待完成 ] 题 2


## 题 3：新裁判的偏差审计

一个新裁判：位置偏差未知，长度偏差未知。用下面的数据估计控制咨询师评分后每 100 字的系数。

In [4]:
from course.judge import SimulatedJudge
from course.data import simulate_counselor_study, DIMENSIONS
st3 = simulate_counselor_study(seed=143, judge=SimulatedJudge(length_bias=0.0, self_preference=0.0, leniency=0.3, noise_sd=0.5, seed=3))
hh = st3["human"]; jj = st3["judge"]; rr = st3["replies"]
d3 = pd.concat([hh[hh.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().rename("human"),
                jj[jj.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().rename("judge")], axis=1).reset_index()
d3 = d3.merge(rr[["item", "system", "n_words"]], on=["item", "system"])
_X = np.column_stack([np.ones(len(d3)), d3.human, d3.n_words / 100])
q3 = None
check("题 3 长度系数", q3, float(np.linalg.lstsq(_X, d3.judge, rcond=None)[0][2]), tol=1e-6)

[ 待完成 ] 题 3 长度系数


*题 3 结论（这个裁判有没有长度偏差？有没有其他偏差？）：*

## 题 4：谁是不认真的评分者

In [5]:
from course.dawid_skene import simulate_raters, dawid_skene, rater_accuracy
L4, t4, a4 = simulate_raters(400, [0.88, 0.6, 0.86, 0.84, 0.5], [0.5, 0.35, 0.15], seed=144)
r4 = dawid_skene(L4, 3)
q4 = None   # 准确率最低的两位评分者下标,升序列表
check("题 4", sorted(q4) if q4 else q4, sorted(np.argsort(rater_accuracy(r4["confusion"], r4["prior"]))[:2].tolist()))

[ 待完成 ] 题 4


## 题 5：写协议摘要

用不超过 200 字写出你的项目裁判验证协议摘要（金标准、指标、门槛、分层、版本锁定）。写完把 `q5` 改成 `True`。

*协议摘要：*

In [6]:
q5 = None
check_true("题 5", q5)

[ 待完成 ] 题 5


## 口述题（录音，每题 90 秒）

1. 用两分钟介绍你的期末项目。
   <details><summary>参考要点</summary>问题（心理健康 LLM 评测缺少经过检验的裁判）、方法（咨询师金标准、七步协议、偏差检验）、你目前的进展、你预期的风险。</details>

In [7]:
summary()

本 notebook 自动批改:0 / 5 通过
